Test


In [ ]:
from google.colab import auth
auth.authenticate_user()
import polars as pl
from datasets import load_dataset
import os

In [ ]:

# --- 1. Cấu hình đường dẫn GCS ---
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET_NAME = "amazon-reviews-lakehouse-data"
listCategory = [
    'Appliances', 'Arts_Crafts_and_Sewing',
    'Automotive', 'Baby_Products', 'Beauty_and_Personal_Care', 'Books',
    'CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry',
    'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food',
    'Health_and_Household', 'Home_and_Kitchen', 'Industrial_and_Scientific',
    'Kindle_Store', 'Luxury_Beauty', 'Magazine_Subscriptions', 'Movies_and_TV',
    'Musical_Instruments', 'Office_Products', 'Patio_Lawn_and_Garden',
    'Pet_Supplies', 'Prime_Pantry', 'Software', 'Sports_and_Outdoors',
    'Subscription_Boxes', 'Tools_and_Home_Improvement', 'Toys_and_Games',
    'Video_Games'
]


In [ ]:
def load_review_data(CATEGORY,batch=50000):
  # Đường dẫn: gs://amazon-reviews-lakehouse-data/bronze-zone/review_data/All_Beauty/
  GCS_SAVE_PATH = f"gs://{BUCKET_NAME}/bronze-zone/review_data/{CATEGORY}"
  # --- 2. Load Dataset từ Hugging Face (Streaming) ---
  dataset = load_dataset(
      "json",
      data_files=f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{CATEGORY}.jsonl",
      split="train",
      streaming=True
  )
  # --- 3. Vòng lặp Batching & Lưu trữ thô ---
  BATCH_SIZE = batch
  buffer = []
  # Danh sách thuộc tính cần giữ lại
  KEEP_KEYS = ['rating', 'text', 'parent_asin', 'timestamp', 'user_id', 'helpful_vote', 'verified_purchase']

  print(f"🚀 Đang đẩy dữ liệu thô {CATEGORY} lên GCS (Bronze Zone)...")

  for i, row in enumerate(dataset):
      # Chỉ trích xuất các cột mục tiêu, giữ nguyên giá trị gốc
      buffer.append({k: row.get(k) for k in KEEP_KEYS})

      if len(buffer) == BATCH_SIZE:
          # A. Tạo Polars DataFrame từ dữ liệu thô
          df_batch = pl.DataFrame(buffer)

          # B. Ghi trực tiếp lên GCS (Không xử lý ETL)
          batch_num = (i + 1) // BATCH_SIZE
          file_name = f"{GCS_SAVE_PATH}/batch_{batch_num}.parquet"

          # Ghi file Parquet với nén mặc định để tiết kiệm dung lượng
          df_batch.write_parquet(file_name, use_pyarrow=True)

          print(f"✅ Đã lưu Batch {batch_num} ({i+1} dòng)")
          buffer = []

  # Xử lý phần dữ liệu còn sót lại
  if buffer:
      final_batch_num = (i // BATCH_SIZE) + 1
      final_file = f"{GCS_SAVE_PATH}/batch_{final_batch_num}_final.parquet"
      pl.DataFrame(buffer).write_parquet(final_file, use_pyarrow=True)
      print(f"✅ Đã lưu Batch cuối cùng.")

  print(f"🏁 Hoàn thành! Dữ liệu thô đã nằm tại: {GCS_SAVE_PATH}")


In [ ]:
def read_review_data(CATEGORY,batch=1000):
  print(f'Du lieu {CATEGORY}')
  path = f"gs://amazon-reviews-lakehouse-data/bronze-zone/review_data/{CATEGORY}/*.parquet"
  lazy_df = pl.scan_parquet(path)
  query = lazy_df.head(batch)
  df = query.collect()
  print(df)

In [ ]:
for CATEGORY in listCategory:
  load_review_data(CATEGORY,200000)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🚀 Đang đẩy dữ liệu thô All_Beauty lên GCS (Bronze Zone)...
✅ Đã lưu Batch 1 (200000 dòng)
✅ Đã lưu Batch 2 (400000 dòng)
✅ Đã lưu Batch 3 (600000 dòng)
✅ Đã lưu Batch cuối cùng.
🏁 Hoàn thành! Dữ liệu thô đã nằm tại: gs://amazon-reviews-lakehouse-data/bronze-zone/review_data/All_Beauty
🚀 Đang đẩy dữ liệu thô Amazon_Fashion lên GCS (Bronze Zone)...
✅ Đã lưu Batch 1 (200000 dòng)
✅ Đã lưu Batch 2 (400000 dòng)
✅ Đã lưu Batch 3 (600000 dòng)
✅ Đã lưu Batch 4 (800000 dòng)
✅ Đã lưu Batch 5 (1000000 dòng)
✅ Đã lưu Batch 6 (1200000 dòng)
✅ Đã lưu Batch 7 (1400000 dòng)
✅ Đã lưu Batch 8 (1600000 dòng)
✅ Đã lưu Batch 9 (1800000 dòng)
✅ Đã lưu Batch 10 (2000000 dòng)
✅ Đã lưu Batch 11 (2200000 dòng)
✅ Đã lưu Batch 12 (2400000 dòng)
✅ Đã lưu Batch cuối cùng.
🏁 Hoàn thành! Dữ liệu thô đã nằm tại: gs://amazon-reviews-lakehouse-data/bronze-zone/review_data/Amazon_Fashion
🚀 Đang đẩy dữ liệu thô Appliances lên GCS (Bronze Zone)...
✅ Đã lưu Batch 1 (200000 dòng)


In [ ]:
for CATEGORY in listCategory:
  read_review_data(CATEGORY,10)

shape: (10, 7)
┌────────┬───────────────┬─────────────┬──────────────┬──────────────┬──────────────┬──────────────┐
│ rating ┆ text          ┆ parent_asin ┆ timestamp    ┆ user_id      ┆ helpful_vote ┆ verified_pur │
│ ---    ┆ ---           ┆ ---         ┆ ---          ┆ ---          ┆ ---          ┆ chase        │
│ f64    ┆ str           ┆ str         ┆ i64          ┆ str          ┆ i64          ┆ ---          │
│        ┆               ┆             ┆              ┆              ┆              ┆ bool         │
╞════════╪═══════════════╪═════════════╪══════════════╪══════════════╪══════════════╪══════════════╡
│ 5.0    ┆ This spray is ┆ B00YQ6X8EO  ┆ 158868772892 ┆ AGKHLEW2SOWH ┆ 0            ┆ true         │
│        ┆ really nice.  ┆             ┆ 3            ┆ NMFQIJGBECAF ┆              ┆              │
│        ┆ It …          ┆             ┆              ┆ 7INQ         ┆              ┆              │
│ 4.0    ┆ This product  ┆ B081TJ8YS3  ┆ 158861585507 ┆ AGKHLEW2SOWH ┆ 1    